## The questions seen from several angles: predefined categories vs. clustered embeddings

Every question is a vector with hundreds of numbers (its embedding). A screen only has two dimensions, so
each picture below is a *projection* that necessarily loses information. To not be fooled by one
projection, the same 400 questions are drawn with several different projection methods.

Each picture has two panels with identical point positions, only the colours differ:

- **Left, predefined categories:** every question is coloured by the Tjenestetilbud it was filed under in ServiceNow.
- **Right, clustered embeddings:** every question is coloured by the theme that HDBSCAN found by looking only at the
  embeddings. Grey points are questions outside every theme. The number is the theme id.

Colours are the same in every picture; the colour keys are in the first section.

No question text is shown anywhere, only question IDs and counts.

## Setup

In [ ]:
import colorsys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patheffects as effects
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
from umap import UMAP

from backend.ai.clustering.question_clustering import (
    CLUSTERING_MODEL_NAME,
    NOISE_LABEL,
    center_embeddings,
    central_question_per_cluster,
    cluster_questions,
    themes_per_question,
)
from backend.ai.similarity_metrics.shared_logic import SERVICE_COLUMN, extract_questions

# Deprecation notices from UMAP, HDBSCAN and tqdm; none of them change the plots
warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd().parent

In [ ]:
questions = extract_questions(REPO_ROOT / "data/raw/serviceNow/serviceNow_questions.xlsx")
embeddings = SentenceTransformer(CLUSTERING_MODEL_NAME).encode(questions["text"].tolist(), normalize_embeddings=True, show_progress_bar=False)

services = questions[SERVICE_COLUMN].fillna("(none)").to_numpy()
themes = cluster_questions(embeddings)
central_rows = central_question_per_cluster(embeddings, themes)

service_sizes = pd.Series(services).value_counts()
service_names = service_sizes.index.tolist()
theme_ids = sorted(int(theme) for theme in np.unique(themes[themes != NOISE_LABEL]))
theme_sizes = {theme: int((themes == theme).sum()) for theme in theme_ids}

print(f"{len(questions)} questions, {embeddings.shape[1]} dimensions each")
print(f"{len(service_names)} Tjenestetilbud (predefined)")
print(f"{len(theme_ids)} themes (clustered), {(themes == NOISE_LABEL).sum()} questions outside every theme")

## Colour keys

In [ ]:
def distinct_colors(count):
    """
    Returns count colours that stay apart: hues are spread by the golden ratio and
    lightness and saturation alternate so that neighbouring hues still differ.
    """
    colors = []
    for index in range(count):
        hue = (index * 0.61803398875) % 1.0
        lightness = (0.38, 0.5, 0.58)[index % 3]
        saturation = (0.85, 0.7)[(index // 3) % 2]
        colors.append(colorsys.hls_to_rgb(hue, lightness, saturation))
    return colors


service_colors = dict(zip(service_names, distinct_colors(len(service_names))))
theme_colors = dict(zip(theme_ids, distinct_colors(len(theme_ids))))


def draw_color_key(axis, title, rows):
    """
    Draws one coloured dot and one line of text per row, top to bottom.
    """
    axis.axis("off")
    axis.set_xlim(0, 1)
    axis.set_ylim(len(rows) + 0.5, -1.5)
    axis.text(0, -0.9, title, fontsize=11, fontweight="bold", va="center")
    for row, (color, text) in enumerate(rows):
        axis.scatter([0.02], [row + 0.2], s=110, color=color, edgecolors="white", linewidths=0.6)
        axis.text(0.05, row + 0.2, text, fontsize=10, va="center")


service_rows = [(service_colors[name], f"{name} ({service_sizes[name]})") for name in service_names]
theme_rows = [
    (theme_colors[theme], f"theme {theme}  |  {theme_sizes[theme]} questions  |  central question {questions.index[central_rows[theme]]}")
    for theme in theme_ids
]

figure, (service_axis, theme_axis) = plt.subplots(1, 2, figsize=(20, 0.36 * max(len(service_rows), len(theme_rows)) + 1.5))
draw_color_key(service_axis, "Tjenestetilbud (questions)", service_rows)
draw_color_key(theme_axis, "Themes (size | central question ID)", theme_rows)
plt.show()

## Projections

All five views start from the same mean-centred embeddings. Centring removes what every question shares
(they are all about HR), which is what the clustering also uses.

In [ ]:
centered = center_embeddings(embeddings)
pca = PCA(n_components=4, random_state=0)
pca_points = pca.fit_transform(centered)

projections = {
    "UMAP, local structure (10 neighbours)": UMAP(n_neighbors=10, min_dist=0.15, metric="cosine", random_state=0).fit_transform(centered),
    "UMAP, global structure (50 neighbours)": UMAP(n_neighbors=50, min_dist=0.3, metric="cosine", random_state=0).fit_transform(centered),
    "t-SNE (perplexity 30)": TSNE(perplexity=30, metric="cosine", init="pca", random_state=0).fit_transform(centered),
    "PCA, components 1 and 2": pca_points[:, [0, 1]],
    "PCA, components 3 and 4": pca_points[:, [2, 3]],
}

kept = pca.explained_variance_ratio_
print(f"PCA components 1-2 keep {kept[:2].sum():.0%} of the variation between questions, components 3-4 keep {kept[2:].sum():.0%}")

In [ ]:
def show_projection(name, points):
    """
    Draws the same points twice: coloured by Tjenestetilbud and coloured by theme.
    """
    figure, (service_axis, theme_axis) = plt.subplots(1, 2, figsize=(18, 8), sharex=True, sharey=True)

    for service_name in service_names:
        members = services == service_name
        service_axis.scatter(*points[members].T, s=40, color=service_colors[service_name], edgecolors="white", linewidths=0.5)
    service_axis.set_title("Coloured by Tjenestetilbud (predefined)", fontsize=12)

    in_theme = themes != NOISE_LABEL
    theme_axis.scatter(*points[~in_theme].T, s=14, color="#d0d0d0", linewidths=0)
    for theme in theme_ids:
        members = themes == theme
        theme_axis.scatter(*points[members].T, s=40, color=theme_colors[theme], edgecolors="white", linewidths=0.5, zorder=3)
        theme_axis.annotate(
            str(theme), np.median(points[members], axis=0), fontsize=10, fontweight="bold", color=theme_colors[theme],
            ha="center", va="center", zorder=5, path_effects=[effects.withStroke(linewidth=3, foreground="white")],
        )
    theme_axis.set_title("Coloured by theme (clustered embeddings), grey = outside every theme", fontsize=12)

    for axis in (service_axis, theme_axis):
        axis.set_xticks([])
        axis.set_yticks([])
    figure.suptitle(name, fontsize=14)
    figure.tight_layout()
    plt.show()

### UMAP, local structure

Keeps each question's closest neighbours together. Good for seeing small groups, but the distances between
groups and the shape of the whole are not reliable.

In [ ]:
show_projection("UMAP, local structure (10 neighbours)", projections["UMAP, local structure (10 neighbours)"])

### UMAP, global structure

Same method but looks at more neighbours, so it keeps more of the overall layout at the cost of local detail.
If a group of questions stays together here as well as in the local view, it is not an artefact of one setting.

In [ ]:
show_projection("UMAP, global structure (50 neighbours)", projections["UMAP, global structure (50 neighbours)"])

### t-SNE

Another neighbour-preserving method with a different way of squeezing the data. The distance between
separate islands carries no meaning here, only which points sit together inside an island.

In [ ]:
show_projection("t-SNE (perplexity 30)", projections["t-SNE (perplexity 30)"])

### PCA, components 1 and 2

A linear view: the two directions along which the questions vary the most. It is not tuned to keep
neighbours together, so groups often overlap, but nothing is distorted.

In [ ]:
show_projection("PCA, components 1 and 2", projections["PCA, components 1 and 2"])

### PCA, components 3 and 4

The next two directions. Showing them makes it visible how much of the structure never appears in the first view.

In [ ]:
show_projection("PCA, components 3 and 4", projections["PCA, components 3 and 4"])

## Questions that belong to more than one theme

Above, every question has exactly one colour: its primary theme, which is the only one to count with.
A question can still be relevant for other themes. For every question the relevance to each theme is
the cosine similarity to the centre of that theme, and the themes within 0.05 of the best one are listed
as "also relevant for".

In the pictures below a black ring marks a question with other relevant themes, and a line in the colour of
each other theme runs from the question to the middle of that theme. Line length is only approximate,
because the 2D view distorts distances.

In [ ]:
memberships = themes_per_question(embeddings, themes)
multi_theme_rows = [row for row, entry in enumerate(memberships) if len(entry) > 1]
placed_count = sum(1 for entry in memberships if entry)

print(f"{len(multi_theme_rows)} of {placed_count} placed questions are also relevant for at least one other theme")

pd.DataFrame({
    "question": [questions.index[row] for row in multi_theme_rows],
    "primary theme": [memberships[row][0][0] for row in multi_theme_rows],
    "primary relevance": [round(memberships[row][0][1], 2) for row in multi_theme_rows],
    "also relevant for (theme: relevance)": [
        ", ".join(f"{theme}: {relevance:.2f}" for theme, relevance in memberships[row][1:])
        for row in multi_theme_rows
    ],
})

In [ ]:
def show_secondary_themes(name, points):
    """
    Draws the themes and, for every question that is also relevant for other themes, a ring
    around the question and a line to the middle of each of those themes.
    """
    figure, axis = plt.subplots(figsize=(12, 9))

    in_theme = themes != NOISE_LABEL
    axis.scatter(*points[~in_theme].T, s=14, color="#d0d0d0", linewidths=0)
    theme_centres = {theme: np.median(points[themes == theme], axis=0) for theme in theme_ids}
    for theme in theme_ids:
        members = themes == theme
        axis.scatter(*points[members].T, s=40, color=theme_colors[theme], edgecolors="white", linewidths=0.5, zorder=3)

    for row in multi_theme_rows:
        for other_theme, _ in memberships[row][1:]:
            axis.plot(*zip(points[row], theme_centres[other_theme]), color=theme_colors[other_theme], linewidth=1.5, zorder=4)
        axis.scatter(*points[row], s=170, facecolors="none", edgecolors="black", linewidths=1.5, zorder=6)

    for theme in theme_ids:
        axis.annotate(
            str(theme), theme_centres[theme], fontsize=10, fontweight="bold", color=theme_colors[theme],
            ha="center", va="center", zorder=7, path_effects=[effects.withStroke(linewidth=3, foreground="white")],
        )
    axis.set_xticks([])
    axis.set_yticks([])
    axis.set_title(f"{name}: ring = question also relevant for other themes, line = to that theme", fontsize=12)
    figure.tight_layout()
    plt.show()

In [ ]:
show_secondary_themes("UMAP, local structure", projections["UMAP, local structure (10 neighbours)"])

In [ ]:
show_secondary_themes("t-SNE", projections["t-SNE (perplexity 30)"])

## How much does each view keep?

Trustworthiness is the share of each question's 10 nearest neighbours in the full embedding that are
still among its nearest neighbours in the picture (1.0 = all kept). It says how far to trust "these
points are close" in each view.

In [ ]:
quality = pd.Series(
    {name: trustworthiness(centered, points, n_neighbors=10) for name, points in projections.items()},
    name="trustworthiness",
).round(2)
quality.sort_values(ascending=False).to_frame()

## How to read the pictures

- Points that are near each other in every view are near each other in the embedding. Points that are
  only near in one view are probably an effect of that projection.
- Left vs. right: if a colour in the left panel is spread over the whole picture, the predefined category
  does not correspond to one region in embedding space.
- A theme that is compact in several views is a stable group. A theme that is spread out in all views is
  held together by directions that none of the 2D views show well.